# NB-2 · Few-step reproduction: does 8-step quality hold on *our* harness?

**Claim under test:** Duo's distilled student generates in 8 steps at quality comparable to its ~1000-step teacher (The Diffusion Duality, ICML 2025). We regenerate with the **authors' own sampler** (ported verbatim from their repo — see `blockspeed/fewstep.py`) and measure with **our calibrated harness** (judge-PPL + entropy, always paired).

**Green means:** `duo-distilled @ 8–16` ≈ `duo teacher @ 1024` on judge-PPL, at healthy entropy (≳5 bits). The expected contrast: `teacher @ 8` should be much worse — that gap is what distillation buys, and it's the foundation of this project's Stage 2.

Runtime: ~20–30 min on a T4 (the 1024-step teacher run dominates). Runtime → Change runtime type → **GPU**.

In [ ]:
%pip -q install --force-reinstall --no-deps git+https://github.com/mccooking/blockspeed.git
%pip -q install einops omegaconf

import blockspeed, torch
print("blockspeed", blockspeed.__version__, "(expect >= 0.2.0)")
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> GPU"
print(torch.cuda.get_device_name(0))

In [ ]:
# Calibrate the instruments in THIS environment before trusting any number.
from blockspeed.harness import self_test

assert self_test(), "harness self-test failed -- do not proceed"

In [ ]:
from blockspeed.fewstep import DUO_DISTILLED, DUO_TEACHER, load_duo, run_config

configs = []

model, tokenizer, device = load_duo(DUO_DISTILLED)
for steps in (8, 16):
    configs.append(run_config(model, tokenizer, "duo-distilled", steps, num_samples=16))
del model
torch.cuda.empty_cache()

In [ ]:
model, tokenizer, device = load_duo(DUO_TEACHER)
configs.append(run_config(model, tokenizer, "duo-teacher", 8, num_samples=16))
configs.append(run_config(model, tokenizer, "duo-teacher", 64, num_samples=16))
configs.append(run_config(model, tokenizer, "duo-teacher", 1024, num_samples=8))  # ~15 min
del model
torch.cuda.empty_cache()

In [ ]:
# Read a couple of samples with your own eyes -- eyes catch what metrics miss.
for cfg in configs:
    print(f"--- {cfg['label']} @ {cfg['num_steps']} steps " + "-" * 40)
    print(cfg["texts"][0][:400].replace("\n", " "), "\n")

In [ ]:
from blockspeed.fewstep import evaluate_configs, plot, save_result

rows = evaluate_configs(configs, judge_model="gpt2-large")
path = save_result(rows, configs, results_dir="results",
                   device_name=torch.cuda.get_device_name(0))
print("saved:", path)
plot(rows)

In [ ]:
# Download the JSON + PNG pair, then commit them to results/ in the repo.
from pathlib import Path
from google.colab import files

for f in sorted(Path("results").glob("fewstep_*")):
    files.download(str(f))